In [1]:
from   osgeo                   import gdal
from   xgboost                 import XGBRegressor
from   joblib                  import load, dump
from   scipy.stats             import kurtosis, skew, iqr
import os
import numpy                   as     np
import pandas                  as     pd
import geopandas               as     gpd
import matplotlib.pyplot       as     plt
import time
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Define Custom Scoring Functions
# Coefficient of Variation
def CV(y_pred):
    return np.std(y_pred, axis=1) / np.mean(y_pred, axis=1)

In [3]:
## CHANGE HERE
## SET BIOMASS POOL: AGBD, BGBD, LBD, OR SBD
## INPUT AND OUTPUT FOLDER, THE OUTPUT FOLDER SHOULD BE ALREADY CREATED
BIOMASS_POOL         = 'BGBD'
Datasets_Directory   = 'C:/_MyOnTheGo/BLS-BIOMASS/ProjectDevelopment/Analysis/Mapping/V60/2_Datasets/'
Regressors_Directory = 'C:/_MyOnTheGo/BLS-BIOMASS/ProjectDevelopment/Analysis/Calibration/V60/Regressors/' + BIOMASS_POOL + '/'
Output_Map_Directory = 'C:/_MyOnTheGo/BLS-BIOMASS/ProjectDevelopment/Analysis/Mapping/V60/3_Maps/'
Stop_Criteria        = 1000
Columns_To_Save      = [BIOMASS_POOL + '_MEAN_FCT-ISA-NMBU', BIOMASS_POOL + '_STANDARDERROR_FCT-ISA-NMBU', BIOMASS_POOL + '_MEDIAN_FCT-ISA-NMBU', BIOMASS_POOL + '_KURTOSIS_FCT-ISA-NMBU', BIOMASS_POOL + '_SKEWNESS_FCT-ISA-NMBU', BIOMASS_POOL + '_IQR_FCT-ISA-NMBU', BIOMASS_POOL + '_CV_FCT-ISA-NMBU', 'geometry']
## END OF CHANGE

# Get Geopackage datasets to be Mapped
Datasets_Paths = [os.path.join(Datasets_Directory, Path) for Path in os.listdir(Datasets_Directory) if BIOMASS_POOL in Path]

In [4]:
## Count time
Time_Process_Start = time.perf_counter()
#---------------------------------------

# Load Regressors
ITR_Counter = 0
Regressors  = []
for Regressor_File in os.listdir(Regressors_Directory):
    print('Loading Regressror #' + str(ITR_Counter+1) + '...')
    Model = XGBRegressor()
    Model.load_model(os.path.join(Regressors_Directory + Regressor_File))
    Regressors.append(Model)
    ITR_Counter += 1
    if ITR_Counter == Stop_Criteria:
        break

#---------------------------------------
# Compute processing time
Time_Process_Ends = time.perf_counter()
Processing_Time   = Time_Process_Ends - Time_Process_Start
# Convert to h:m:s
hours, rem       = divmod(Processing_Time, 3600)
minutes, seconds = divmod(rem, 60)
print(f"Elapsed time: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")

Loading Regressror #1...
Loading Regressror #2...
Loading Regressror #3...
Loading Regressror #4...
Loading Regressror #5...
Loading Regressror #6...
Loading Regressror #7...
Loading Regressror #8...
Loading Regressror #9...
Loading Regressror #10...
Loading Regressror #11...
Loading Regressror #12...
Loading Regressror #13...
Loading Regressror #14...
Loading Regressror #15...
Loading Regressror #16...
Loading Regressror #17...
Loading Regressror #18...
Loading Regressror #19...
Loading Regressror #20...
Loading Regressror #21...
Loading Regressror #22...
Loading Regressror #23...
Loading Regressror #24...
Loading Regressror #25...
Loading Regressror #26...
Loading Regressror #27...
Loading Regressror #28...
Loading Regressror #29...
Loading Regressror #30...
Loading Regressror #31...
Loading Regressror #32...
Loading Regressror #33...
Loading Regressror #34...
Loading Regressror #35...
Loading Regressror #36...
Loading Regressror #37...
Loading Regressror #38...
Loading Regressror #3

In [5]:
# Count time
Time_Process_Start = time.perf_counter()
#---------------------------------------

# Load data in chunks
# Define Chunck Size
#Chunk_Size  = 1_000_000
Chunk_Size  = 500_000
Start       = 0
Chunks      = dict()
# Iterate all Geopackages to be processed
for Dataset_Path in Datasets_Paths:
    # Count time
    Time_Area_Start = time.perf_counter()
    #---------------------------------------
    Area = Dataset_Path.split('/')[-1].split('_')[0]
    print(Dataset_Path)
    # Start Reading Chunks
    Chunks[Area] = []
    Start       = 0
    while True:
        # Count Iteration time
        Time_Iteration_Start = time.perf_counter()
        #---------------------------------------
        
        print(Start/Chunk_Size)
        # Get data chunk
        Data_Chunk = gpd.read_file(Dataset_Path, rows=slice(Start, Start + Chunk_Size))
        # If the Chunk does not contain any data, the end of the dataset was reached
        if Data_Chunk.empty:
            break
        # Get data for the regression
        Data_X = Data_Chunk.drop(columns = [BIOMASS_POOL, 'geometry'])
        # Make predictions
        Data_Preds = np.empty((len(Data_Chunk), Stop_Criteria), dtype=np.float32)
        for i, Model in enumerate(Regressors):
            print(f"Iteration {i+1}...")
            Data_Preds[:, i] = Model.predict(Data_X)
        # Compute each Sample Prediction & Uncertainty
        Final_Mean = np.mean(Data_Preds, axis=1, keepdims=True).astype('float32')
        Final_Uncertainty = np.std(Data_Preds, axis=1, keepdims=True).astype('float32')
        Final_Median = np.median(Data_Preds, axis=1, keepdims=True).astype('float32')
        Final_Kurtosis = kurtosis(Data_Preds, axis=1, bias=False).astype('float32')[:, None]
        Final_Skewness = skew(Data_Preds, axis=1, bias=False).astype('float32')[:, None]
        Final_IQR = iqr(Data_Preds, axis=1).astype('float32')[:, None]
        Final_CV = (Final_Uncertainty / Final_Mean).astype('float32')
        # Save V1 on the Footprints Data Chunk
        Data_Chunk[BIOMASS_POOL + '_MEAN_FCT-ISA-NMBU'         ] = Final_Mean
        Data_Chunk[BIOMASS_POOL + '_STANDARDERROR_FCT-ISA-NMBU'] = Final_Uncertainty
        Data_Chunk[BIOMASS_POOL + '_MEDIAN_FCT-ISA-NMBU'       ] = Final_Median
        Data_Chunk[BIOMASS_POOL + '_KURTOSIS_FCT-ISA-NMBU'     ] = Final_Kurtosis
        Data_Chunk[BIOMASS_POOL + '_SKEWNESS_FCT-ISA-NMBU'     ] = Final_Skewness
        Data_Chunk[BIOMASS_POOL + '_IQR_FCT-ISA-NMBU'          ] = Final_IQR
        Data_Chunk[BIOMASS_POOL + '_CV_FCT-ISA-NMBU'           ] = Final_CV    
        # Save Chunk
        Chunks[Area].append(Data_Chunk)
        # Move to next chunk
        Start       += Chunk_Size
       
        #---------------------------------------
        # Compute processing time
        Time_Iteration_Ends = time.perf_counter()
        Iteration_Time   = Time_Iteration_Ends - Time_Iteration_Start
        # Convert to h:m:s
        hours, rem       = divmod(Iteration_Time, 3600)
        minutes, seconds = divmod(rem, 60)
        print(f"Elapsed time: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")    
            
    #---------------------------------------
    # Compute processing time
    Time_Area_Ends   = time.perf_counter()
    Area_Time        = Time_Area_Ends - Time_Area_Start
    # Convert to h:m:s
    hours, rem       = divmod(Area_Time, 3600)
    minutes, seconds = divmod(rem, 60)
    print(f"Elapsed time: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")

#---------------------------------------
# Compute processing time
Time_Process_Ends = time.perf_counter()
Processing_Time   = Time_Process_Ends - Time_Process_Start
# Convert to h:m:s
hours, rem       = divmod(Processing_Time, 3600)
minutes, seconds = divmod(rem, 60)
print(f"Elapsed time: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")

C:/_MyOnTheGo/BLS-BIOMASS/ProjectDevelopment/Analysis/Mapping/V60/2_Datasets/Extremadura_BGBD_FieldDataset.gpkg
0.0
Iteration 1...
Iteration 2...
Iteration 3...
Iteration 4...
Iteration 5...
Iteration 6...
Iteration 7...
Iteration 8...
Iteration 9...
Iteration 10...
Iteration 11...
Iteration 12...
Iteration 13...
Iteration 14...
Iteration 15...
Iteration 16...
Iteration 17...
Iteration 18...
Iteration 19...
Iteration 20...
Iteration 21...
Iteration 22...
Iteration 23...
Iteration 24...
Iteration 25...
Iteration 26...
Iteration 27...
Iteration 28...
Iteration 29...
Iteration 30...
Iteration 31...
Iteration 32...
Iteration 33...
Iteration 34...
Iteration 35...
Iteration 36...
Iteration 37...
Iteration 38...
Iteration 39...
Iteration 40...
Iteration 41...
Iteration 42...
Iteration 43...
Iteration 44...
Iteration 45...
Iteration 46...
Iteration 47...
Iteration 48...
Iteration 49...
Iteration 50...
Iteration 51...
Iteration 52...
Iteration 53...
Iteration 54...
Iteration 55...
Iteration 56.

In [6]:
# Count time
Time_Process_Start = time.perf_counter()
#---------------------------------------

for Area in Chunks.keys():
     # Count Iteration time
    Time_Area_Start = time.perf_counter()
    #---------------------------------------
    
    # Get Area to address
    print(Area)
    # Save Geopandas dataframe into geopackage
    Output_Geo_Path           = Output_Map_Directory + Area + '_' + BIOMASS_POOL + '_Map.gpkg'
    Final_Geopandas_Dataframe = gpd.GeoDataFrame(pd.concat(Chunks[Area], ignore_index=True))
    Final_Geopandas_Dataframe = Final_Geopandas_Dataframe[Columns_To_Save]
    Final_Geopandas_Dataframe.to_file(Output_Geo_Path, driver="GPKG")
    # Generate raster files
    for Column in Columns_To_Save:
        if Column == 'geometry':
            continue
        print(Column)
        Output_Map_Path = Output_Map_Directory + Area + '_' + Column + '_Map.tif'
        gdal.Rasterize(
            Output_Map_Path,
            Output_Geo_Path,
            attribute=Column,
            xRes=25,
            yRes=25,
            outputBounds=Final_Geopandas_Dataframe.total_bounds,
            noData=-9999,
            initValues=-9999
        )
    
    #---------------------------------------
    # Compute processing time
    Time_Area_Ends   = time.perf_counter()
    Area_Time        = Time_Area_Ends - Time_Area_Start
    # Convert to h:m:s
    hours, rem       = divmod(Area_Time, 3600)
    minutes, seconds = divmod(rem, 60)
    print(f"Elapsed time: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")

#---------------------------------------
# Compute processing time
Time_Process_Ends = time.perf_counter()
Processing_Time   = Time_Process_Ends - Time_Process_Start
# Convert to h:m:s
hours, rem       = divmod(Processing_Time, 3600)
minutes, seconds = divmod(rem, 60)
print(f"Elapsed time: {int(hours):02d}:{int(minutes):02d}:{seconds:05.2f}")

Extremadura
BGBD_MEAN_FCT-ISA-NMBU
BGBD_STANDARDERROR_FCT-ISA-NMBU
BGBD_MEDIAN_FCT-ISA-NMBU
BGBD_KURTOSIS_FCT-ISA-NMBU
BGBD_SKEWNESS_FCT-ISA-NMBU
BGBD_IQR_FCT-ISA-NMBU
BGBD_CV_FCT-ISA-NMBU
Elapsed time: 00:18:36.16
Elapsed time: 00:18:36.16
